In [58]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.sql import text
import warnings
import sys
import os # Esto sube un nivel en las carpetas para encontrar la raíz del proyecto
from dotenv import load_dotenv

sys.path.append(os.path.abspath(".."))

import src.db_builder as db

# Verificar dónde está buscando el .env
print("Directorio actual:", os.getcwd())

# Cargar y comprobar variables
load_dotenv("../.env")

Directorio actual: c:\Users\fabih\Desktop\CREATOR\data\data_e-commerce\e-commerce-operations-insights\notebooks


True

In [59]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

In [60]:
df_data = pd.read_csv("../files/processed/dataco_supply_chain.csv", sep=None, engine="python", encoding="latin-1")

In [61]:
df_data.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,delivery_date,order_date,order_time
42924,DEBIT,2,1,89.17,181.98,Late delivery,1,9,Cardio Equipment,Rosemead,EE. UU.,Patricia,9614,Frazier,Consumer,CA,2472 Noble Butterfly Dell,91770,3,Footwear,34.065777,-118.082642,Durgapur,India,9614,26824,191,18.00,0.09,67210,99.99,0.49,2,199.98,181.98,89.17,South Asia,COMPLETE,191,9,Nike Men's Free 5.0+ Running Shoe,99.99,First Class,01-29-2016,01-27-2016,13:17
7103,DEBIT,6,2,-43.05,245.98,Late delivery,1,43,Camping & Hiking,Caguas,Puerto Rico,Rose,3145,Smith,Consumer,PR,4436 Merry By-pass,725,7,Fan Shop,18.212431,-66.370567,Bolingbrook,Estados Unidos,3145,34390,957,54.00,0.18,85917,299.98,-0.18,1,299.98,245.98,-43.05,US Center,COMPLETE,957,43,Diamondback Women's Serene Classic Comfort Bi,299.98,Second Class,05-22-2016,05-16-2016,23:59
91129,PAYMENT,4,4,-44.63,57.59,Shipping on time,0,17,Cleats,Caguas,Puerto Rico,Michelle,5201,Keith,Corporate,PR,8189 Clear Key,725,4,Apparel,18.268795,-66.370552,Elazig,TurquÃ­a,5201,43128,365,2.40,0.04,107743,59.99,-0.78,1,59.99,57.59,-44.63,West Asia,PENDING_PAYMENT,365,17,Perfect Fitness Perfect Rip Deck,59.99,Standard Class,09-25-2016,09-21-2016,13:18
111342,DEBIT,3,2,-37.96,207.42,Late delivery,1,46,Indoor/Outdoor Games,Caguas,Puerto Rico,Sara,2365,Smith,Consumer,PR,7940 Noble Field,725,7,Fan Shop,18.272745,-66.370575,Lisichansk,Ucrania,2365,50112,1014,42.48,0.17,125270,49.98,-0.18,5,249.90,207.42,-37.96,Eastern Europe,COMPLETE,1014,46,O'Brien Men's Neoprene Life Vest,49.98,Second Class,01-04-2017,01-01-2017,12:06
35978,DEBIT,2,1,8.71,263.98,Late delivery,1,43,Camping & Hiking,Lynwood,EE. UU.,Amanda,10701,Smith,Consumer,CA,437 Heather Quail Concession,90262,7,Fan Shop,33.146004,-117.135658,Villeurbanne,Francia,10701,10914,957,36.00,0.12,27329,299.98,0.03,1,299.98,263.98,8.71,Western Europe,COMPLETE,957,43,Diamondback Women's Serene Classic Comfort Bi,299.98,First Class,06-11-2015,06-09-2015,07:18


In [62]:
locations = df_data[['latitude', 'longitude']].drop_duplicates().reset_index(drop=True)
locations['location_id'] = locations.index + 1
locations = locations[['location_id', 'latitude', 'longitude']]

In [63]:
locations.tail(5)

,location_id,latitude,longitude
11830,11831,18.275261,-66.037056
11831,11832,18.223066,-66.037056
11832,11833,18.240482,-66.037064
11833,11834,18.261297,-66.037056
11834,11835,18.242485,-66.370514


In [64]:
departments = df_data[["department_id", "department_name"]].drop_duplicates(subset=["department_id"]).reset_index(drop=True)

In [65]:
departments.sample(5)

,department_id,department_name
0,2,Fitness
2,5,Golf
3,3,Footwear
6,10,Technology
9,11,Pet Shop


In [66]:
categories = df_data[['category_id', 'category_name']].drop_duplicates(subset=['category_id']).reset_index(drop=True)

In [67]:
categories.sample(5)

,category_id,category_name
16,64,Computers
39,69,Health and Beauty
30,5,Lacrosse
2,29,Shop By Sport
1,17,Cleats


In [68]:
products = df_data[["product_card_id", "product_name", "product_price", "category_id"]].drop_duplicates(subset=["product_card_id"]).reset_index(drop=True)

In [69]:
products.sample(5)

,product_card_id,product_name,product_price,category_id
86,216,Yakima DoubleDown Ace Hitch Mount 4-Bike Rack,189.00,10
44,835,Bridgestone e6 Straight Distance NFL Carolina,31.99,37
64,1348,CDs of rock,11.29,61
16,403,Nike Men's CJ Elite 2 TD Football Cleat,129.99,18
35,116,Nike Men's Comfort 2 Slide,44.99,6


In [70]:
df_data = pd.merge(df_data, locations, on=['latitude', 'longitude'], how='left')

In [71]:
customers = df_data[['customer_id', 'customer_fname', 'customer_lname', 
                    'customer_segment', 'customer_street', 'location_id', 
                    'customer_city', 'customer_state', 'customer_zipcode', 'customer_country'
                    ]].drop_duplicates(subset=['customer_id']).reset_index(drop=True)

In [72]:
customers.head(5)

,customer_id,customer_fname,customer_lname,customer_segment,customer_street,location_id,customer_city,customer_state,customer_zipcode,customer_country
0,20755,Cally,Holloway,Consumer,5365 Noble Nectar Island,1,Caguas,PR,725,Puerto Rico
1,19492,Irene,Luna,Consumer,2679 Rustic Loop,2,Caguas,PR,725,Puerto Rico
2,19491,Gillian,Maldonado,Consumer,8510 Round Bear Gate,3,San Jose,CA,95125,EE. UU.
3,19490,Tana,Tate,Home Office,3200 Amber Bend,4,Los Angeles,CA,90027,EE. UU.
4,19489,Orli,Hendricks,Corporate,8671 Iron Anchor Corners,5,Caguas,PR,725,Puerto Rico


In [ ]:
orders = df_data[['order_id', 'customer_id', 'department_id', 'type', 'order_date', 
                  'order_time', 'order_status', 'order_city', 'order_country', 
                  'order_region', 'shipping_mode', 'delivery_date', 'days_for_shipping_real', 
                  'days_for_shipment_scheduled', 'delivery_status', 
                  'late_delivery_risk']].drop_duplicates(subset=['order_id']).reset_index(drop=True)

In [75]:
orders.tail()

,order_id,customer_id,department_id,type,order_date,order_time,order_status,order_city,order_country,order_region,shipping_mode,delivery_date,days_for_shipping_real,days_for_shipment_scheduled,delivery_status,late_delivery_risk
65747,26773,5857,7,CASH,01-26-2016,19:25,CLOSED,Ho Chi Minh City,Vietnam,Southeast Asia,First Class,01-28-2016,2,1,Late delivery,1
65748,26463,9230,7,PAYMENT,01-22-2016,06:49,PENDING_PAYMENT,Ezhou,China,Eastern Asia,Standard Class,01-26-2016,4,4,Shipping on time,0
65749,26383,4618,7,CASH,01-21-2016,02:47,CLOSED,Tawau,Malasia,Southeast Asia,Second Class,01-25-2016,4,2,Late delivery,1
65750,26327,989,7,DEBIT,01-20-2016,07:10,ON_HOLD,Semarang,Indonesia,Southeast Asia,Standard Class,01-23-2016,3,4,Advance shipping,0
65751,26118,6251,7,TRANSFER,01-17-2016,05:56,SUSPECTED_FRAUD,Ratlam,India,South Asia,Standard Class,01-21-2016,4,4,Shipping canceled,0


In [77]:
order_items = df_data[['order_item_id', 'order_id', 'product_card_id', 'order_item_quantity',
                    'order_item_product_price', 'order_item_discount', 
                    'order_item_discount_rate', 'sales', 'order_item_total', 
                    'order_item_profit_ratio', 'benefit_per_order']].drop_duplicates(subset=['order_item_id']).reset_index(drop=True)

In [79]:
order_items.sample(5)

,order_item_id,order_id,product_card_id,order_item_quantity,order_item_product_price,order_item_discount,order_item_discount_rate,sales,order_item_total,order_item_profit_ratio,benefit_per_order
114031,7847,3152,1014,4,49.98,23.99,0.12,199.92,175.93,-0.13,-21.99
87210,38856,15545,502,1,50.00,6.50,0.13,50.00,43.50,0.31,13.49
80461,74267,29685,1073,1,199.99,20.00,0.10,199.99,179.99,0.48,86.40
125471,3861,1546,1004,1,399.98,68.00,0.17,399.98,331.98,0.06,20.91
111844,120891,48332,1014,2,49.98,19.99,0.20,99.96,79.97,0.05,4.00
